# Fig 3 C & D — IDI distributions

**3C** — 12 ms-bin IDI histogram (real fish dyad, top) + agents (bottom)  
**3D** — Power-law fit (real fish dyad, top) + agents (bottom)

Agent data: `2fish_m1a1k1_uniform_wide` — `derived/per_env_ep_agent_step.pkl`  
Real fish data: `real_fish_data/dyad_ipis_ms_50kHz.csv.gz`

Derived pkl must already exist (run pipeline first if missing).

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
RUN_DIR = "/srv/marl/satsingh/marl_fish/NEW/foraging/Dyn_F00_Kb_For_S1"

# Set to a directory path to save PDFs there; None = inline display only
OUT_DIR = None

In [ ]:
import sys, os, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import powerlaw

sys.path.insert(0, '/home/satsingh/kr/mfrefactor/onpolicy/custom/fish')
os.chdir('/home/satsingh/kr/mfrefactor/onpolicy/custom/fish')

import analysis_idi as m_idi; importlib.reload(m_idi)
from analysis_idi import (
    _plot_12ms_histogram, _plot_powerlaw_fit, compute_idis
)
from analysis_style import TIME_STEP_MS, set_style, panel, save, inline_display

_HERE = os.path.abspath('real_fish_data')
_DYAD_CSV = os.path.join(_HERE, 'dyad_ipis_ms_50kHz.csv.gz')

wide_dir = os.path.join(RUN_DIR, 'evals', '2fish_m1a1k1_uniform_wide')
step_pkl = os.path.join(wide_dir, 'derived', 'per_env_ep_agent_step.pkl')
assert os.path.exists(step_pkl), f"Missing: {step_pkl}"
print(f"Loading {step_pkl} ...")
df = pd.read_pickle(step_pkl)
print(f"  {len(df):,} rows")

agent_idis = compute_idis(df)
print(f"  {len(agent_idis):,} agent IDIs, range {agent_idis.min():.1f}–{agent_idis.max():.1f} ms")

In [ ]:
# Load real fish dyad IPIs (trim 12–1200 ms, snap to 12 ms grid)
from real_fish_data.plot_real_fish_idi import load_dyad_idis, _trim_and_discretize

dyad_raw = load_dyad_idis()
dyad_idis = _trim_and_discretize(dyad_raw, min_ms=12, max_ms=1200)
print(f"Dyad IPIs: {len(dyad_idis):,} after trim (raw: {len(dyad_raw):,})")

## Fig 3C — 12 ms-bin IDI histogram

In [ ]:
# Real fish (top sub-panel)
set_style()
with inline_display():
    fig, ax = panel(2.0, 1.25)
    _plot_12ms_histogram(ax, dyad_idis, title="Real")
    plt.tight_layout(pad=0.3)
    if OUT_DIR:
        save(fig, os.path.join(OUT_DIR, 'real_fish_dyad_ipi_histogram.pdf'))

In [ ]:
# Agent (bottom sub-panel)
set_style()
with inline_display():
    fig, ax = panel(2.0, 1.25)
    _plot_12ms_histogram(ax, agent_idis, title="Agent")
    plt.tight_layout(pad=0.3)
    if OUT_DIR:
        save(fig, os.path.join(OUT_DIR, 'idi_histogram_12ms.pdf'))

## Fig 3D — Power-law fit

In [ ]:
# Real fish (top sub-panel)
set_style()
with inline_display():
    fig, ax = panel(2.0, 1.25)
    _plot_powerlaw_fit(ax, dyad_idis, "dyad", title="Real", bin_width_ms=12)
    plt.tight_layout(pad=0.3)
    if OUT_DIR:
        save(fig, os.path.join(OUT_DIR, 'real_fish_dyad_ipi_powerlaw.pdf'))

In [ ]:
# Agent (bottom sub-panel)
set_style()
with inline_display():
    fig, ax = panel(2.0, 1.25)
    _plot_powerlaw_fit(
        ax, agent_idis, "Agents", title="Agent",
        bin_width_ms=TIME_STEP_MS,
    )
    plt.tight_layout(pad=0.3)
    if OUT_DIR:
        save(fig, os.path.join(OUT_DIR, 'idi_powerlaw.pdf'))